# 03 Portfolio Comparison
Thin notebook: baseline vs proposed folder comparison via ReplayAnalyzer.

In [1]:
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

cwd = os.getcwd()
candidates = [cwd, os.path.dirname(cwd)]
REPO = next((p for p in candidates if os.path.exists(os.path.join(p, 'src'))), cwd)
SRC = os.path.join(REPO, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import margin_analysis
importlib.reload(margin_analysis)
import replay_analyzer
importlib.reload(replay_analyzer)
from replay_analyzer import default_analyzer_for_repo

In [2]:
analyzer = default_analyzer_for_repo(REPO)
comparison_result = analyzer.compare_folders(
    baseline_folder=os.path.join(REPO, 'data', 'baseline'),
    proposed_folder=os.path.join(REPO, 'data', 'proposed')
)

tables = comparison_result['tables']

print('Configuration Comparison')
display(tables['configuration'])

print('Summary Comparison')
display(tables['performance'])

print('Drawdown Comparison')
display(tables['drawdown'])

print('Margin Comparison')
display(tables['margin'])

print('Per-Pair Comparison')
display(tables['per_pair'])

# Margin-analysis comparison (canonical events + threshold summaries)
baseline_ma = comparison_result['baseline'].margin_analysis()
proposed_ma = comparison_result['proposed'].margin_analysis()

baseline_events = baseline_ma.events_df()
proposed_events = proposed_ma.events_df()

baseline_summary = baseline_ma.portfolio_summary_dict(baseline_events)
proposed_summary = proposed_ma.portfolio_summary_dict(proposed_events)

summary_metrics = sorted(set(baseline_summary.keys()) | set(proposed_summary.keys()))
margin_analysis_summary = []
for metric in summary_metrics:
    b = baseline_summary.get(metric)
    p = proposed_summary.get(metric)
    delta = (p - b) if isinstance(b, (int, float)) and isinstance(p, (int, float)) else None
    margin_analysis_summary.append({
        'Metric': metric,
        'Baseline': b,
        'Proposed': p,
        'Delta': delta,
    })

print('Margin Analysis Summary Comparison')
display(pd.DataFrame(margin_analysis_summary))

print('Canonical Margin Events Comparison')
canonical_event_comparison = pd.DataFrame([
    {
        'Metric': 'canonical_event_count',
        'Baseline': len(baseline_events),
        'Proposed': len(proposed_events),
        'Delta': len(proposed_events) - len(baseline_events),
    },
    {
        'Metric': 'worst_event_min_margin_level',
        'Baseline': baseline_events['min_margin_level'].min() if not baseline_events.empty else None,
        'Proposed': proposed_events['min_margin_level'].min() if not proposed_events.empty else None,
        'Delta': (
            (proposed_events['min_margin_level'].min() - baseline_events['min_margin_level'].min())
            if (not baseline_events.empty and not proposed_events.empty)
            else None
        ),
    },
])
display(canonical_event_comparison)

baseline_event_summary = baseline_ma.threshold_summary_df(baseline_events).rename(columns={
    'event_count': 'event_count_baseline',
    'total_minutes_below': 'total_minutes_below_baseline',
    'longest_event_minutes_below': 'longest_event_minutes_below_baseline',
    'worst_margin_level': 'worst_margin_level_baseline',
})

proposed_event_summary = proposed_ma.threshold_summary_df(proposed_events).rename(columns={
    'event_count': 'event_count_proposed',
    'total_minutes_below': 'total_minutes_below_proposed',
    'longest_event_minutes_below': 'longest_event_minutes_below_proposed',
    'worst_margin_level': 'worst_margin_level_proposed',
})

margin_event_comparison = baseline_event_summary.merge(
    proposed_event_summary,
    on='threshold',
    how='outer'
).sort_values('threshold', ascending=False)

for lhs, rhs, out in [
    ('event_count_baseline', 'event_count_proposed', 'event_count_delta'),
    ('total_minutes_below_baseline', 'total_minutes_below_proposed', 'total_minutes_below_delta'),
    ('longest_event_minutes_below_baseline', 'longest_event_minutes_below_proposed', 'longest_event_minutes_below_delta'),
    ('worst_margin_level_baseline', 'worst_margin_level_proposed', 'worst_margin_level_delta'),
]:
    if lhs in margin_event_comparison.columns and rhs in margin_event_comparison.columns:
        margin_event_comparison[out] = margin_event_comparison[rhs] - margin_event_comparison[lhs]

print('Margin Event Comparison By Threshold (Aggregated From Canonical Events)')
display(margin_event_comparison)

baseline_pair = baseline_ma.pair_summary_df(baseline_events)
proposed_pair = proposed_ma.pair_summary_df(proposed_events)

if not baseline_pair.empty or not proposed_pair.empty:
    pair_margin_comparison = baseline_pair.merge(
        proposed_pair,
        on='pair',
        suffixes=(' (Baseline)', ' (Proposed)'),
        how='outer'
    ).sort_values('pair')

    print('Margin Pair Contribution Comparison')
    display(pair_margin_comparison)

No XLSX found for EURGBP, skipping
No XLSX found for GBPUSD, skipping
No XLSX found for USDCHF, skipping
No XLSX found for EURGBP, skipping
No XLSX found for GBPUSD, skipping
No XLSX found for USDCHF, skipping


Configuration Comparison


,Pair,Risk (Baseline),TP (Baseline),Grid (Baseline),Max Trades (Baseline),Trades (Baseline),Initial Balance (Baseline),Median Lot (Baseline),Risk (Proposed),TP (Proposed),Grid (Proposed),Max Trades (Proposed),Trades (Proposed),Initial Balance (Proposed),Median Lot (Proposed)
0,CADJPY,1.4,15,60,3,309,5000.0,0.09,1.4,15,60,3,309,5000.0,0.09
1,EURUSD,1.2,23,29,4,575,5000.0,0.10,2.8,15,54,4,359,5000.0,0.28
2,NZDCHF,1.8,10,31,5,315,5000.0,0.12,1.8,10,31,5,315,5000.0,0.12


Summary Comparison


,Metric,Baseline,Proposed,Delta
0,Initial Balance,5000.000000,5000.000000,0.000000
1,Final Balance,34316.975400,24045.385000,-10271.590400
2,Profit,29316.975400,19045.385000,-10271.590400
3,Average Monthly Growth,4.132916,3.397246,-0.735670
4,CAGR,64.818900,50.465900,-14.353000
5,Profit Factor,2.683277,2.087913,-0.595364
6,Recovery Factor,5.462242,2.213614,-3.248628
7,Total Trades,2398.000000,1966.000000,-432.000000
8,Winning Trades,890.000000,834.000000,-56.000000
9,Win Rate,37.114262,42.421160,5.306898


Drawdown Comparison


... (9 outputs left out)

Margin Event Comparison By Threshold (Aggregated From Canonical Events)


,threshold,event_count_baseline,total_minutes_below_baseline,longest_event_minutes_below_baseline,worst_margin_level_baseline,event_count_proposed,total_minutes_below_proposed,longest_event_minutes_below_proposed,worst_margin_level_proposed,event_count_delta,total_minutes_below_delta,longest_event_minutes_below_delta,worst_margin_level_delta
3,300.0,110,229775,86340,137.43,159,586689,87465,108.11,49,356914,1125,-29.32
2,200.0,14,161921,86340,137.43,13,451334,87465,108.11,-1,289413,1125,-29.32
1,150.0,2,33363,31065,137.43,3,187127,87465,108.11,1,153764,56400,-29.32
0,100.0,0,0,0,NaN,0,0,0,NaN,0,0,0,NaN


Margin Pair Contribution Comparison


,pair,avg_used_margin (Baseline),max_used_margin (Baseline),avg_used_margin_contribution_pct (Baseline),peak_used_margin_contribution_pct (Baseline),avg_floating_pnl (Baseline),worst_floating_pnl (Baseline),avg_open_positions (Baseline),max_open_positions (Baseline),margin_events_participated (Baseline),...,avg_used_margin (Proposed),max_used_margin (Proposed),avg_used_margin_contribution_pct (Proposed),peak_used_margin_contribution_pct (Proposed),avg_floating_pnl (Proposed),worst_floating_pnl (Proposed),avg_open_positions (Proposed),max_open_positions (Proposed),margin_events_participated (Proposed),margin_events_as_largest_contributor (Proposed)
0,CADJPY,1050.50,5131.78,34.68,100.0,-436.24,-3304.41,1.38,3,100,...,963.95,4383.66,28.24,100.0,-404.27,-3111.92,1.38,3,128,46
1,EURUSD,469.11,13940.70,15.47,100.0,-32.59,-5201.74,1.10,4,99,...,1447.01,7805.78,33.98,100.0,-604.48,-7718.27,1.46,4,145,87
2,NZDCHF,1101.40,3142.61,48.23,100.0,-382.74,-3408.85,2.20,5,110,...,1014.54,2819.37,36.19,100.0,-355.73,-3211.60,2.20,5,157,26


In [3]:
comparison_result['charts']['equity_comparison'].show()
comparison_result['baseline'].plot_margin().show()
comparison_result['proposed'].plot_margin().show()